# B04b: Wavelet Analysis (Quick Version)

**Duration**: ~30 minutes  
**Prerequisites**: B04a, B01

> 💡 **Quick Version**: Imports pre-built functions from `src/wavelets.py`. See the full tutorial for implementation details.


## Table of Contents

1. [Introduction — Wavelets vs STFT](#section-1-introduction)
2. [The Morlet Wavelet](#section-2-morlet)
3. [Wavelet Transform](#section-3-transform)
4. [Phase from Wavelets](#section-4-phase)
5. [Edge Effects](#section-5-edge-effects)
6. [Exercises](#section-6-exercises)
7. [Summary](#summary)
8. [External Resources](#external-resources)
9. [Discussion Questions](#discussion-questions)


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import sys
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import hilbert

src_path = Path.cwd().parent.parent.parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from wavelets import (
    create_morlet_wavelet,
    wavelet_convolution,
    compute_wavelet_transform,
    compute_wavelet_power,
    compute_wavelet_phase,
    compute_adaptive_cycles,
    compute_edge_samples,
)
from colors import COLORS

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

np.random.seed(42)
fs = 256
print("Imports loaded ✓")


---

<a id="section-1-introduction"></a>
## 1. Introduction — Wavelets vs STFT

**Key insight**: Wavelets provide *adaptive* time-frequency resolution.

| | STFT | Morlet Wavelets |
|--|------|----------------|
| **Resolution** | Fixed for all frequencies | High freq → better time res; Low freq → better freq res |
| **Window** | Fixed-length analysis window | Frequency-dependent window length |
| **Phase** | Complex output → phase extractable | Complex output → phase extractable |
| **EEG use** | General spectrograms | Connectivity analysis, event-related |

The Morlet wavelet at frequency $f$:
$$\psi_f(t) = A \cdot e^{-t^2/2\sigma_t^2} \cdot e^{i 2\pi f t}$$


In [ ]:
# Visualize Morlet wavelets at different frequencies
frequencies = [4, 10, 30]
fig, axes = plt.subplots(len(frequencies), 2, figsize=(12, 7))

for row, freq in enumerate(frequencies):
    wavelet, t_wav = create_morlet_wavelet(freq, fs, n_cycles=5, return_time=True)
    axes[row, 0].plot(t_wav, np.real(wavelet), color=COLORS["signal_1"], lw=1.5, label="Real")
    axes[row, 0].plot(t_wav, np.abs(wavelet),  color=COLORS["negative"], lw=2, ls="--", label="Envelope")
    axes[row, 0].set_title(f"{freq} Hz wavelet (n_cycles=5)", fontweight="bold")
    axes[row, 0].legend(loc="upper right", fontsize=9)
    axes[row, 0].grid(True, alpha=0.3)
    
    # Frequency response
    fft_wav = np.fft.fft(wavelet, n=len(wavelet)*4)
    freqs_wav = np.fft.fftfreq(len(wavelet)*4, 1/fs)
    pos_mask = freqs_wav > 0
    axes[row, 1].plot(freqs_wav[pos_mask], np.abs(fft_wav)[pos_mask],
                      color=COLORS["high_sync"], lw=1.5)
    axes[row, 1].set_xlim(0, freq*3)
    axes[row, 1].axvline(freq, color="gray", ls="--", alpha=0.7)
    axes[row, 1].set_title(f"Frequency response (peak: {freq} Hz)", fontweight="bold")
    axes[row, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Frequency (Hz)")
plt.tight_layout()
plt.show()
print("→ Higher frequency wavelets are shorter in time but narrower in frequency")


---

<a id="section-3-transform"></a>
## 3. Wavelet Transform

The wavelet transform convolves the signal with wavelets at multiple frequencies.


In [ ]:
# Generate EEG-like signal with frequency modulations
t = np.arange(0, 6.0, 1/fs)
alpha_burst = (0.3 + 0.7 * ((t > 1) & (t < 3)).astype(float)) * np.sin(2 * np.pi * 10 * t)
gamma_burst = (0.3 + 0.8 * ((t > 3.5) & (t < 4.5)).astype(float)) * np.sin(2 * np.pi * 40 * t)
eeg = alpha_burst + gamma_burst + 0.3 * np.random.randn(len(t))

# Compute wavelet transform
frequencies = np.linspace(2, 50, 30)  # 30 frequency steps
n_cycles = compute_adaptive_cycles(frequencies, min_cycles=3, max_cycles=7)

tf_power = compute_wavelet_power(eeg, fs, frequencies, n_cycles_array=n_cycles)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(t, eeg, color=COLORS["signal_1"], lw=0.7)
axes[0].set_ylabel("Amplitude (a.u.)")
axes[0].set_title("EEG Signal (Alpha + Gamma bursts)", fontweight="bold")
axes[0].grid(True, alpha=0.3)

pcm = axes[1].pcolormesh(t, frequencies, tf_power, cmap='hot', shading='gouraud',
                          norm=LogNorm(vmin=tf_power.max()*0.01, vmax=tf_power.max()))
plt.colorbar(pcm, ax=axes[1], label="Power (a.u.)")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Wavelet Time-Frequency Power", fontweight="bold")
axes[1].axhline(10, color="white", lw=0.8, ls="--", alpha=0.7, label="Alpha (10 Hz)")
axes[1].axhline(40, color="cyan", lw=0.8, ls="--", alpha=0.7, label="Gamma (40 Hz)")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()


---

<a id="section-4-phase"></a>
## 4. Phase from Wavelets

Wavelets give complex output → extract phase for connectivity analysis.


In [ ]:
# Extract phase at alpha frequency using wavelet
tf_phase = compute_wavelet_phase(eeg, fs, frequencies=[10.0])  # shape: (1, n_times)
alpha_phase = tf_phase[0]  # instantaneous phase at 10 Hz

# Compare wavelet phase vs Hilbert phase
from filtering import bandpass_filter
alpha_filtered = bandpass_filter(eeg, 8, 13, fs)
hilbert_phase = np.angle(hilbert(alpha_filtered))

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(t, hilbert_phase, color=COLORS["signal_1"], lw=1, label="Hilbert phase")
axes[0].plot(t, alpha_phase, color=COLORS["negative"], lw=1, ls="--", label="Wavelet phase (10 Hz)")
axes[0].set_ylabel("Phase (rad)")
axes[0].legend()
axes[0].set_title("Hilbert vs Wavelet Phase — Alpha Band", fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Phase difference
phase_diff = np.angle(np.exp(1j * (alpha_phase - hilbert_phase)))
axes[1].plot(t, phase_diff, color=COLORS["high_sync"], lw=1, alpha=0.7)
axes[1].set_ylabel("Phase diff (rad)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Phase Difference (near zero → methods agree)", fontweight="bold")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

corr = np.corrcoef(np.cos(alpha_phase), np.cos(hilbert_phase))[0, 1]
print(f"Correlation between methods: {corr:.3f} (methods agree well)")


---

<a id="section-5-edge-effects"></a>
## 5. Edge Effects

Wavelet convolution has edge artifacts at signal boundaries — always mask them.


In [ ]:
# Compute edge samples to mask
edge_n_samples = {freq: compute_edge_samples(freq, fs, n_cyc) 
                  for freq, n_cyc in zip(frequencies, n_cycles)}

# Show edge effects at a few frequencies
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
selected_freqs = [4, 10, 30]

for ax, target_freq in zip(axes, selected_freqs):
    freq_idx = np.argmin(np.abs(frequencies - target_freq))
    actual_freq = frequencies[freq_idx]
    tf_complex = compute_wavelet_transform(eeg, fs, [actual_freq], n_cycles_array=[n_cycles[freq_idx]])
    power_trace = np.abs(tf_complex[0]) ** 2
    
    n_edge = compute_edge_samples(actual_freq, fs, n_cycles[freq_idx])
    
    ax.plot(t, power_trace, color=COLORS["signal_1"], lw=1.5, label=f"{actual_freq:.0f} Hz power")
    ax.axvspan(t[0], t[n_edge], color="red", alpha=0.2, label=f"Edge artifact ({n_edge} samples)")
    ax.axvspan(t[-n_edge], t[-1], color="red", alpha=0.2)
    ax.legend(loc="upper right", fontsize=9)
    ax.set_ylabel("Power")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
plt.suptitle("Wavelet Edge Effects — Always Mask Boundary Samples", fontweight="bold")
plt.tight_layout()
plt.show()
print("→ Lower frequencies require masking MORE samples (wider wavelet)")


---

<a id="section-6-exercises"></a>
## 6. Exercises

### 🎯 Exercise 1: n_cycles Trade-off

**Difficulty**: 🟡 Medium  
**Task**: Compute wavelet power at 10 Hz with n_cycles = 3, 5, and 10. Compare temporal vs frequency resolution.

```python
for n_cyc in [3, 5, 10]:
    power = compute_wavelet_power(eeg, fs, frequencies=[10.0], n_cycles_array=[n_cyc])
    # Plot and compare sharpness of alpha burst onset
```

### 🎯 Exercise 2: Time-Resolved PLV

**Difficulty**: 🔴 Hard  
**Task**: Create two 10 Hz signals with the same phase for t=[1,3] and random phases elsewhere. Compute time-resolved PLV using wavelet phases.

```python
# Hint:
# 1. extract wavelet phase at 10 Hz for both signals
# 2. compute phase difference: delta_phi = phase1 - phase2
# 3. PLV in window: |mean(exp(1j * delta_phi))| over sliding window
```


---

<a id="summary"></a>
## 7. Summary

### Key Concepts

| Concept | Key Point |
|---------|-----------|
| **Morlet wavelet** | Gaussian × complex sinusoid — localized in both time and frequency |
| **n_cycles** | Controls time-frequency resolution: more cycles → better freq res, worse time res |
| **Adaptive n_cycles** | Use more cycles for low frequencies, fewer for high |
| **Phase extraction** | `compute_wavelet_phase()` → complex output → `np.angle()` |
| **Edge effects** | `compute_edge_samples()` → always mask boundary regions |

### Functions from `src/wavelets.py`

| Function | Purpose |
|----------|---------|
| `create_morlet_wavelet` | Build a single Morlet wavelet |
| `compute_wavelet_transform` | Full CWT across multiple frequencies |
| `compute_wavelet_power` | `|CWT|²` power matrix |
| `compute_wavelet_phase` | `angle(CWT)` phase matrix |
| `compute_adaptive_cycles` | Frequency-dependent n_cycles selection |
| `compute_edge_samples` | Number of samples to mask at boundaries |


---

<a id="external-resources"></a>
## 8. External Resources

### 📚 Scientific References

- **Morlet, J. et al. (1982)** - *Wave propagation and sampling theory* — Original wavelet paper
- **Tallon-Baudry, C. & Bertrand, O. (1999)** - *Oscillatory gamma activity in humans* — Morlet wavelets in EEG
- **Cohen, M.X. (2014)** - *Analyzing Neural Time Series Data* — Chapters 12–14

### 🎧 NotebookLM Resources

*(NotebookLM links will be added once this notebook is finalized)*


---

<a id="discussion-questions"></a>
## 9. Discussion Questions

1. **Wavelets vs Hilbert**: Both methods extract instantaneous phase. When would you prefer wavelets over the Hilbert transform? When would you prefer Hilbert?

2. **n_cycles selection**: The `compute_adaptive_cycles` function uses more cycles for low frequencies. What is the neurophysiological justification for this choice?

3. **Edge effects and trial length**: You have 1-second epochs and want to analyze 4 Hz (delta) oscillations. How many cycles do you need? Is 1 second enough data?
